Features:
- add moving average by worker / item / process

split

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_data

In [2]:
data = load_data('data_2')

data['duration'] = (data.time_to - data.time_from).dt.total_seconds()

# Duration Prediction Part

## Feature Engineering

In [3]:
data['weekday'] = data.time_from.dt.weekday
data['day'] = data.time_from.dt.day

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class StoreFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, history_cutoff_date):
        self.history_cutoff_date = pd.to_datetime(history_cutoff_date)
        self.ma_dictionary = {}
        

    def fit(self, X, y=None):
        df = X.copy()
        df['duration'] = (df['time_to'] - df['time_from']).dt.total_seconds()

        daily_stats = df.assign(date=df['time_from'].dt.normalize())\
            .groupby(['worker_id', 'item_id', 'process_name', 'date'], as_index=False)['duration']\
            .mean()

        daily_stats['ma_duration'] = daily_stats.sort_values('date')\
            .groupby(['worker_id', 'item_id', 'process_name'])['duration']\
            .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())

            self.ma_dictionary = daily_stats.set_index(['worker_id', 'item_id', 'process_name', 'date'])['ma_duration'].to_dict()
        
        return self

    def transform(self, X):
        df = X.copy()
        
        # Create features from date
        df['weekday'] = df['time_from'].dt.weekday
        df['day'] = df['time_from'].dt.day
        df['date'] = df['time_from'].dt.normalize()
        
        # Map MA duration by worker - item - process - date
        df['ma_duration'] = df.apply(
            lambda row: self.ma_dictionary.get((row['worker_id'], row['item_id'], row['process_name'], row['date'])),
            axis=1
        )
        
        # Return
        categorical_features = ['worker_id', 'item_id', 'process_name'] 
        numerical_features = ['weekday', 'day', 'ma_duration']
        X_out = df[categorical_features + numerical_features].copy()
        for f in categorical_features:
            X_out[f] = X_out[f].astype('category')
        
        return X_out

In [26]:
daily_stats = df.assign(date=df['time_from'].dt.normalize())\
    .groupby(['worker_id', 'item_id', 'process_name', 'date'], as_index=False)['duration']\
    .mean()

daily_stats['ma_duration'] = daily_stats.sort_values('date')\
    .groupby(['worker_id', 'item_id', 'process_name'])['duration']\
    .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())

ma_dictionary = daily_stats.set_index(['worker_id', 'item_id', 'process_name', 'date'])['ma_duration'].to_dict()

In [27]:
daily_stats

,worker_id,item_id,process_name,date,duration,ma_duration
0,worker_1,item_1,collection,2025-07-01,26.6230,NaN
1,worker_1,item_1,collection,2025-07-02,24.7895,26.623000
2,worker_1,item_1,collection,2025-07-06,29.3245,25.706250
3,worker_1,item_1,collection,2025-07-07,22.6330,26.912333
4,worker_1,item_1,collection,2025-07-09,20.8855,25.842500
...,...,...,...,...,...,...
2912,worker_5,item_5,transfer,2025-09-13,56.8190,61.075357
2913,worker_5,item_5,transfer,2025-09-14,60.6890,59.630214
2914,worker_5,item_5,transfer,2025-09-17,62.6940,59.705643
2915,worker_5,item_5,transfer,2025-09-19,62.9950,59.788929
